# VIGIL Notebook 3: Threshold Tuning
## Experimental Analysis of Decision Thresholds

This notebook explores:
- Effect of threshold on precision/recall
- Precision-Recall trade-offs
- False Positive Rate analysis
- Optimal threshold selection for different use cases

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

project_root = Path('.').resolve()
sys.path.insert(0, str(project_root))
from src.utils.metrics import MetricsComputer

print('[INFO] Threshold tuning notebook initialized')

In [ ]:
# Generate synthetic verification scores and ground truth
np.random.seed(42)

scores = np.random.uniform(0.1, 0.9, size=100)
ground_truth = np.random.choice(['TRUST', 'WARNING'], size=100, p=[0.6, 0.4])

print(f'Generated {len(scores)} verification scores')
print(f'Ground truth: {sum(gt=="TRUST" for gt in ground_truth)} TRUST, {sum(gt=="WARNING" for gt in ground_truth)} WARNING')

In [ ]:
# Sweep over thresholds
print('[STEP] Simulating threshold sweep...')

thresholds = np.linspace(0.1, 0.9, 20)
metrics_by_threshold = {}

for thresh in thresholds:
    predictions = [MetricsComputer.make_decision(s, thresh) for s in scores]
    try:
        prec, rec = MetricsComputer.compute_precision_recall(predictions, ground_truth.tolist())
        fpr = MetricsComputer.compute_fpr(predictions, ground_truth.tolist())
        accuracy = MetricsComputer.compute_accuracy(predictions, ground_truth.tolist())
        
        metrics_by_threshold[float(thresh)] = {
            'precision': prec,
            'recall': rec,
            'fpr': fpr,
            'accuracy': accuracy
        }
    except:
        pass

print(f'✓ Sweep complete: {len(metrics_by_threshold)} thresholds analyzed')

In [ ]:
# Plot precision-recall curve and FPR analysis
precisions = [m['precision'] for m in metrics_by_threshold.values()]
recalls = [m['recall'] for m in metrics_by_threshold.values()]
fprs = [m['fpr'] for m in metrics_by_threshold.values()]
accuracies = [m['accuracy'] for m in metrics_by_threshold.values()]
thresh_list = sorted(metrics_by_threshold.keys())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Precision-Recall
axes[0, 0].plot(recalls, precisions, 'o-', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Recall')
axes[0, 0].set_ylabel('Precision')
axes[0, 0].set_title('Precision-Recall Curve')
axes[0, 0].grid(True, alpha=0.3)

# FPR vs Threshold
axes[0, 1].plot(thresh_list, fprs, 's-', linewidth=2, markersize=6, color='red')
axes[0, 1].set_xlabel('Threshold')
axes[0, 1].set_ylabel('False Positive Rate')
axes[0, 1].set_title('FPR vs Decision Threshold')
axes[0, 1].grid(True, alpha=0.3)

# Accuracy vs Threshold
axes[1, 0].plot(thresh_list, accuracies, '^-', linewidth=2, markersize=6, color='green')
axes[1, 0].set_xlabel('Threshold')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Accuracy vs Decision Threshold')
axes[1, 0].grid(True, alpha=0.3)

# All metrics
axes[1, 1].plot(thresh_list, precisions, 'o-', label='Precision', linewidth=2)
axes[1, 1].plot(thresh_list, recalls, 's-', label='Recall', linewidth=2)
axes[1, 1].plot(thresh_list, [1-f for f in fprs], '^-', label='1-FPR', linewidth=2)
axes[1, 1].set_xlabel('Threshold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('All Metrics vs Threshold')
axes[1, 1].legend(loc='best')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./results/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Threshold analysis plots saved')

In [ ]:
# Find optimal thresholds for different objectives
best_precision_idx = np.argmax(precisions)
best_recall_idx = np.argmax(recalls)
best_accuracy_idx = np.argmax(accuracies)
best_fpr_idx = np.argmin(fprs)

print('Optimal thresholds for different objectives:')
print(f'  For max precision: {thresh_list[best_precision_idx]:.2f} (precision={precisions[best_precision_idx]:.3f})')
print(f'  For max recall: {thresh_list[best_recall_idx]:.2f} (recall={recalls[best_recall_idx]:.3f})')
print(f'  For max accuracy: {thresh_list[best_accuracy_idx]:.2f} (accuracy={accuracies[best_accuracy_idx]:.3f})')
print(f'  For min FPR: {thresh_list[best_fpr_idx]:.2f} (FPR={fprs[best_fpr_idx]:.3f})')